In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from dataset import init_data_loader
from utils import test_args, load_model, generate_prediction_scores

In [ ]:
@torch.no_grad()
def generate_prediction_scores(model, test_dataloader, args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    model.to(device)
    model.eval()
    ls = []

    with tqdm(total=len(test_dataloader)) as pbar: # -args.seq_length+1
        for i, (char_with_label, _) in enumerate(test_dataloader):
            char = char_with_label[:,:,:-1].to(device)
            returns = char_with_label[:,:,-1].to(device)

            predictions = model.prediction(char.float())
            ls.append(predictions.detach().cpu())
            pbar.update(1)
            
    ls = torch.cat(ls, dim=0)
    indicies = test_dataloader.dataset.sampler.get_index()
    multi_index = pd.MultiIndex.from_tuples(indicies, names=["datetime","instrument"])
    ls = pd.DataFrame(ls.numpy(), index=multi_index, columns=['score'])
    return ls

In [ ]:
args = test_args(run_name='VAE-Revision2',
                num_factor=96, 
                normalize=False, 
                select_feature=False,
                num_latent=158,
                num_portfolio=128,
                hidden_size=64,
                use_qlib=False)

factorVAE = load_model(args)
model_name = './best_models/VAE-Revision2_factor_96_hdn_64_port_128_seed_42.pt'
factorVAE.load_state_dict(torch.load(model_name))

In [ ]:
# dataset = pd.read_pickle('data/csi_data.pkl').iloc[:, :159]
dataset = pd.read_pickle('data/csi_300_inference.pkl').iloc[:, :159]
dataset.rename(columns={dataset.columns[-1]: 'LABEL0'}, inplace=True) 
test_dataloader = init_data_loader(dataset,
                                    shuffle=False, 
                                    step_len= 20 , 
                                    start= '2023-01-01', 
                                    end= '2025-12-31',)

In [ ]:
output = generate_prediction_scores(factorVAE, test_dataloader, args)

In [ ]:
output = pd.merge(output, dataset['LABEL0'],  right_index=True, left_index=True)

In [ ]:
from pprint import pprint
import qlib.contrib.report as qcr
import qlib
import pandas as pd
from qlib.utils.time import Freq
from qlib.utils import flatten_dict
from qlib.backtest import backtest, executor
from qlib.contrib.evaluate import backtest_daily
from qlib.contrib.evaluate import risk_analysis
from qlib.contrib.strategy import TopkDropoutStrategy

# init qlib
qlib.init(provider_uri='/workspace/AAAI_VQVAE/qlib_data/cn_data')

CSI300_BENCH = "SH000300"
FREQ = "day"
STRATEGY_CONFIG = {
    "topk": 50,
    "n_drop": 10,
    # pred_score, pd.Series
    "signal": output,
}

EXECUTOR_CONFIG = {
    "time_per_step": "day",
    "generate_portfolio_metrics": True,
}

backtest_config = {
    "start_time": "2023-01-01",
    "end_time": "2025-12-31",
    "account": 100000000,
    "benchmark": CSI300_BENCH,
    "exchange_kwargs": {
        "freq": FREQ,
        "limit_threshold": 0.095,
        "deal_price": "close",
        "open_cost": 0.0005,
        "close_cost": 0.0015,
        "min_cost": 5,
    },
}
# strategy object
strategy_obj = TopkDropoutStrategy(**STRATEGY_CONFIG)
# executor object
executor_obj = executor.SimulatorExecutor(**EXECUTOR_CONFIG)

# backtest
portfolio_metric_dict, indicator_dict = backtest(executor=executor_obj, strategy=strategy_obj, **backtest_config)
analysis_freq = "{0}{1}".format(*Freq.parse(FREQ))

# backtest info
report_normal_df, positions_normal = portfolio_metric_dict.get(analysis_freq)

In [ ]:
import qlib.contrib.report as qcr
from qlib.contrib.report import analysis_position, analysis_model
print(qcr.GRAPH_NAME_LIST)
analysis_position.report_graph(report_normal_df)

In [ ]:
analysis = dict()
analysis["excess_return_without_cost"] = risk_analysis(
    report_normal_df["return"] - report_normal_df["bench"], freq=analysis_freq
)
analysis["excess_return_with_cost"] = risk_analysis(
    report_normal_df["return"] - report_normal_df["bench"] - report_normal_df["cost"], freq=analysis_freq
)

analysis_df = pd.concat(analysis)
pprint(analysis_df)

In [ ]:
from utils import RankIC
RankIC(output.dropna(axis=0), column1='score', column2='LABEL0')